# 9.11 不生成一段話，也能回答問題嗎？


客服材料寫「使用者要查卡片申請進度」，程式只想知道該開哪個頁面。自由回答可能寫出「幫他看看卡片吧」，但程式還得把文字轉成動作。這次先把允許的答案列清楚，讓模型直接對每個候選給分，再用[9.9](https://birdhackor.github.io/tiny-perceptron-vlm/9.9.html)學過的 softmax 轉成機率。

一次判斷有三部分：**state（狀態材料）**是使用者說過的內容與系統已知事實；**question（問題）**指定現在要判斷什麼；**candidates（候選）**說明可選答案的意義。下表是本節的例子；分數是人工填的示意值，機率由下面的程式算出，沒有訓練模型：

| 狀態 | 問題 | 候選意義 | 人工分數 | 機率 |
| --- | --- | --- | ---: | ---: |
| 要查卡片申請進度 | 下一步服務？ | 查卡片進度 | 2.0 | 0.844 |
| 同上 | 同上 | 更改地址 | −1.0 | 0.042 |
| 同上 | 同上 | 以上皆不適合 | 0.0 | 0.114 |

真正的模型會把「狀態＋問題＋某個候選意義」編成向量，經同一個**共享評分器**得到一個分數；每個候選都用同一套評分方法，[9.12](https://birdhackor.github.io/tiny-perceptron-vlm/9.12.html)會說明為什麼要這樣做。本節還沒有模型，下面的程式直接用表中的人工分數，看一次判斷怎麼從分數走到動作：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

judgment = {
    "state": "使用者要查卡片申請進度",
    "question": "下一步服務？",
    "candidates": ["查卡片進度", "更改地址", "以上皆不適合"],
}
scores = torch.tensor([2.0, -1.0, 0.0])  # 人工示意分數，代替訓練好的共享評分器
probability = scores.softmax(-1)
for meaning, p in zip(judgment["candidates"], probability.tolist()):
    print(meaning, round(p, 3))
actions = {"查卡片進度": "開啟卡片進度頁", "更改地址": "開啟地址表單", "以上皆不適合": "轉給真人客服"}
best = judgment["candidates"][int(probability.argmax())]
print("程式依規則採取的動作：", actions[best])

`judgment`把一次判斷的三部分放進同一個字典。`scores`的三個數依序對應三個候選；`softmax(-1)`把它們轉成總和為1的機率，`.tolist()`轉回普通清單，`zip`（[1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html)）把候選名稱和機率一對一配起來逐行印出：0.844、0.042、0.114。這三個數四捨五入到三位，相加仍是1。`argmax`找出最高機率的位置0，`int(...)`把它轉成普通整數，再取回候選名稱「查卡片進度」。最後查`actions`字典，印出「開啟卡片進度頁」：哪個候選對應哪個動作，是程式事先寫好的規則。

分數的產生是模型工作，實際開頁或修改資料則是應用程式工作；高分不直接授權執行任何操作。

這類介面稱為**有型別的機率判斷**：型別規定答案能長什麼樣子，例如選項分布、yes 機率或有限等級分布。它讓程式不必再猜自由文字的格式，也能保留「第二可能」等資訊：本例第二高的是「以上皆不適合」（0.114），只回一句自由文字就看不出這一點。這個需要和一般聊天不同，所以我們在誠實與校準之後學它。

外部也有這類服務。**Jev** 是 TypeSafe 公司提供的判斷服務：程式在一次呼叫裡送入資料與問題，拿回可以直接讀取、不必解析文字的有型別結果。官方把問題分成三種型別，正好對應上面三種答案形狀：

- **Choice**：從列出的選項中挑一個，回傳選項分布，就是本節的例子。
- **Noul**：問一句話是否為真，回傳一個 yes 機率，[9.13](https://birdhackor.github.io/tiny-perceptron-vlm/9.13.html)會用到。
- **Score**：依事先寫好的等級規準評分，回傳有次序的有限等級分布，[9.14](https://birdhackor.github.io/tiny-perceptron-vlm/9.14.html)會用到。

這三種型別是任務與輸出的約定，不是一張足以推得內部層數、注意力或訓練細節的架構圖。本課在第19章自己訓練一個同類的**機率判斷器**：從隨機初始化的參數學候選評分，不宣稱重製 Jev 的專有訓練方法。它的基礎仍是條件分類，也就是給定狀態與問題，替候選分類；多種問題、用意義描述的候選、校準與使用約定合起來，才組成完整服務。

練習：把程式的`"question"`改成`"是否已有訂單編號？"`，其他行都不動。先預測輸出完全不變，仍印出「開啟卡片進度頁」，再執行核對。這個輸出有意義嗎？沒有：新問題的答案應是有或沒有，原來的候選卻是服務項目，人工分數也是替舊問題填的。真正的評分器會讀`question`，這個示意程式不讀，所以看不出錯。問題決定要讀哪種資訊，候選也須跟著問題定義；下一節讓這個依賴變得可檢查。

來源：[TypeSafe introduction](https://docs.typesafe.ai/introduction)（讀取2026-10-18）。
